# Understanding the experience

The scientific question is : Can a circular steering task quantify sensorimotor integration in persons with stroke?

To do so, MouseReMoCo software is used. 18 volunteers are doing the tasks (9 healthy volunteers and 9 persons with stroke). The task is to try to follow the curve of a circle with a mouse. They perform the task 6 times (2 hands x 3 repetitions). 

What are the two files?

001MoDe_R1.csv: data on the movement (position x, y of the mouse, timestamp and mouseintarget)
001MoDe_R1.marker.csv: data on the recording

What is measured?

The Index of Effective Performance (IPe, in bit/s) which measure the effectiveness of visuomotor feedback.


# 1 - Read header

### Import libraries

In [31]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle # for drawing shapes
import os # for changing the working directory

# for interactive plots in Jupyter notebooks
%matplotlib widget

if os.getcwd().endswith("notebooks"):
    os.chdir("..") # for running the notebook from the notebooks folder, change to the parent directory

### Display the header

In [32]:
def read_header(fname):
    """Read the configuration line of a MouseReMoCo file and return a dictionary."""
    with open(fname, "r") as f:
        config_line = f.readline().strip()   # first line of the file, without trailing newline

    params = {}
    for pair in config_line.split(";"):      # cut at each semicolon
        if pair.strip() == "":
            continue                         # ignore empty pairs
        name, value = pair.split(" ", 1)     # cut at the first space : name | value
        try:
            params[name] = float(value)      # if it's a number -> number
        except ValueError:
            params[name] = value             # otherwise we keep the text
    return params

In [33]:
data_file = os.path.join("data", "001MoDe_R1.csv")
marker_file = os.path.join("data", "001MoDe_R1.marker.csv")

params = read_header(data_file)

print(f"Header line information in '{data_file}':")
for name, value in params.items():
    print(f"  {name}: {value}")

print("Same header in both files:", params == read_header(marker_file))

Header line information in 'data\001MoDe_R1.csv':
  screenWidth: 1104.0
  screenHeight: 661.0
  cornerX: 302.0
  cornerY: 80.0
  centerX: 552.0
  centerY: 330.0
  externalRadius: 250.0
  internalRadius: 170.0
  borderRadius: 1.0
  cursorRadius: 16.0
  indexOfDifficulty: 28.00696429476858
  taskRadius: 209.5
  taskTolerance: 47.0
  borderColor: java.awt.Color[r=255,g=255,b=255]
  backgroundColor: java.awt.Color[r=0,g=0,b=0]
  cursorColorRecord: java.awt.Color[r=255,g=0,b=0]
  cursorColorWait: java.awt.Color[r=255,g=255,b=0]
  autoStart: 3600.0
  cycleMaxNumber: 6.0
  cycleDuration: 20.0
  software: LSL-mouse
  version: 1.2.0rc5
  task: CircularTarget
  isWithLSL: false
Same header in both files: True


The same header is in both files. It contains information on the screen dimensions (first four lines), on the elements of the task in pixels:

- centerX: horizontal coordinate of the center of the circle

- centerY: vertical coordinate of the center of the circle

- externalRadius: external radius of the circle

- internalRadius: internal radius of the circle (because the circle has two lines)

- borderRadius: width of the border (1 px)

- cursorRadius: radius of the cursor (16 px)

and:

- indexOfDifficulty: how difficult is the task

- taskRadius: radius of the trajectory that the participants need to follow (the value is between internalRadius and externalRadius) (pixels)

- taskTolerance: allowed width for the cursor center (47 px)

It also contains information about the colors of the border, background and cursor. 

It informs us also about:

- cycleMaxNumber: maximum number of cycles (6)

- cycleDuration: in seconds

and, the information about the software. 

# 2 - Read 001MoDe_R1.marker.csv and 001MoDe_R1.csv

### Read markers (001MoDe_R1.marker.csv)

In [34]:
# Define a function that reads events from the marker file.
def read_markers(fname):
    # Create an empty list to store the markers found.
    markers = []

    # Open the file whose path is stored in fname.
    with open(fname, "r") as f:
        # Read the file one line at a time.
        for line in f:
            # Remove spaces and the newline character.
            line = line.strip()
            # Split the line into a maximum of three parts.
            parts = line.split(",", 2)

            # Check that the line has three parts
            # and that the second part is a number.
            if len(parts) == 3 and parts[1].isdigit():
                # Store the timestamp.
                timestamp = int(parts[1])
                # Store the event name or text.
                event = parts[2]
                # Add the timestamp and event to the list.
                markers.append((timestamp, event))

    # Return the list of markers.
    return markers


# Call the function with the path stored in marker_file.
markers = read_markers(marker_file)

# Display the name of the file being read.
print(f"Markers in '{marker_file}':")

# Go through each marker in the list.
for timestamp, event in markers:
    # Display the timestamp and the corresponding event.
    print(f"{timestamp}: {event}")

Markers in 'data\001MoDe_R1.marker.csv':
1616776712325: KeyTyped=32 DoCycleChange
1616776712331: DoCycleChange:DoStartCycleTimedSequence
1616776712333: DoCycleChange:DoRecord RecordDone=0 PauseDone=0 ToDo=6
1616776732336: DoCycleChange:DoPause RecordDone=1 PauseDone=0 ToDo=6
1616776752340: DoCycleChange:DoRecord RecordDone=1 PauseDone=1 ToDo=6
1616776772351: DoCycleChange:DoPause RecordDone=2 PauseDone=1 ToDo=6
1616776792372: DoCycleChange:DoRecord RecordDone=2 PauseDone=2 ToDo=6
1616776812386: DoCycleChange:DoPause RecordDone=3 PauseDone=2 ToDo=6
1616776832390: DoCycleChange:DoRecord RecordDone=3 PauseDone=3 ToDo=6
1616776852404: DoCycleChange:DoPause RecordDone=4 PauseDone=3 ToDo=6
1616776872419: DoCycleChange:DoRecord RecordDone=4 PauseDone=4 ToDo=6
1616776892433: DoCycleChange:DoPause RecordDone=5 PauseDone=4 ToDo=6
1616776898364: KeyTyped=113 WINDOW_CLOSING
1616776898367: 


The big numbers stand for the timestamp that is to say the time when a recording is done. It will be confirmed later but it is in milliseconds. The elements on the left:

- DoCycleChange: when there is a change 

- DoStartCycleTimedSequence: start of the cycle 

- DoRecord: start of a recording phase

- DoPause: start of pause phase

- RecordDone: total number of recordings done

- PauseDone: total number of pauses 

- ToDo=6: total number of recordings to be done

- KeyTyped=113 WINDOW_CLOSING: the keytouch corresponds to the action of closing the window. 

The lines displayed correspond to markers. Markers are used to display the time an event happens and describe the event.

The first line seems to indicate the start of the experience.
However, the first recording starts at line 3. The first timestamp of the data file is 1616776712329 (t = 0).

We can see that the total number of recordings is 5, although 6 was expected.

### Display the metrics

In [35]:
# Read the marker file.
with open(marker_file, "r") as f:
    lines = f.readlines()

# Find the beginning of the results table.
start = next(
    i for i, line in enumerate(lines)
    if line.strip().startswith("Var ,")
)

# Display the results table.
for line in lines[start:]:
    print(line.rstrip())

    Var , nLaps ,      Re ,      Te ,   error ,  MT/lap , IDe/lap ,      Be ,     IPe ,
   unit ,   lap ,   pixel ,   pixel ,       % ,   s/lap , bit/lap ,  double ,   bit/s ,
 Theory ,  1.00 ,  209.50 ,   47.00 ,    3.88 ,         ,         ,    1.00 ,         ,
 Rec001 ,-10.85 ,  213.87 ,   91.79 ,    3.90 ,    1.84 ,   14.64 ,    1.95 ,    7.94 ,
 Rec002 ,-11.97 ,  209.85 ,   34.46 ,    0.74 ,    1.59 ,   38.27 ,    0.73 ,   24.04 ,
 Rec003 ,-13.26 ,  209.43 ,   42.42 ,    2.81 ,    1.45 ,   31.02 ,    0.90 ,   21.40 ,
 Rec004 ,-12.83 ,  209.46 ,   37.14 ,    0.13 ,    1.49 ,   35.43 ,    0.79 ,   23.72 ,
 Rec005 ,-14.74 ,  210.89 ,   43.88 ,    2.84 ,    1.30 ,   30.20 ,    0.93 ,   23.20 ,
 Rec006 ,  0.00 ,    0.00 ,    0.00 ,     NaN ,     NaN ,     NaN ,    0.00 ,    0.00 ,


The results displayed confirm that the data for the sixth recording are not usable.

The metrics are the following: 

- nLaps: number of laps

- Re: effective radius

- Te: effective tolerance

- error: percentage of the lap outside the target

- MT/lap: MovementTime per lap

- IDe/lap: Index of difficulty per lap

- Be: Te / 47 (effective tolerance divided by the task tolerance)

- IPe: Effective performance index

### Read 001MoDe_R1.csv

In [36]:
# Read and display the data file
with open(data_file, "r") as f:
    lines = f.readlines()

print(f"Data table from '{data_file}':")

# Display the column names and first rows
for line in lines[3:15]:
    print(line.rstrip())

# Load the numerical data
data = np.loadtxt(data_file, delimiter=",", skiprows=4)

# Remove rows where the timestamp is 0
data = data[data[:, 0] != 0]

print("\nData loaded successfully.")
print("Data shape:", data.shape)

Data table from 'data\001MoDe_R1.csv':
timestamp,mouseX,mouseY,mouseInTarget
1616776712329,501,554,1
1616776712351,487,551,1
1616776712356,484,550,1
1616776712359,481,549,1
1616776712366,478,548,1
1616776712372,475,547,1
1616776712375,473,546,1
1616776712382,470,545,1
1616776712384,467,544,1
1616776712389,464,543,1
1616776712396,461,542,1

Data loaded successfully.
Data shape: (19588, 4)


### Extract the start and end timestamps of each trial

In [37]:
# Extract the start and end timestamps of each trial
starts = []
ends = []

for timestamp, event in markers:
    if "DoRecord" in event:
        starts.append(timestamp)
    elif "DoPause" in event:
        ends.append(timestamp)

# Check that the number of starts and ends are equal
print("Number of starts:", len(starts))
print("Number of ends:", len(ends))

if len(starts) != len(ends):
    print("Error: The number of starts and ends are not equal.")
else:
    print("The number of starts and ends are equal.")

Number of starts: 5
Number of ends: 5
The number of starts and ends are equal.


### Record times (ms)   

In [38]:
print("record times (milliseconds since the Unix epoch, 00:00:00 UTC on January 1, 1970):")
for i in range(min(len(starts), len(ends))):
    duration = (ends[i] - starts[i]) / 1000
    print(f"{starts[i]} to {ends[i]} ({duration:.3f} s)")

record times (milliseconds since the Unix epoch, 00:00:00 UTC on January 1, 1970):
1616776712333 to 1616776732336 (20.003 s)
1616776752340 to 1616776772351 (20.011 s)
1616776792372 to 1616776812386 (20.014 s)
1616776832390 to 1616776852404 (20.014 s)
1616776872419 to 1616776892433 (20.014 s)


# 3 - Split the data into records

### Time in seconds and separate variables

In [39]:
# Convert time to seconds (t = 0 at the first data point)
t = (data[:, 0] - data[0, 0]) / 1000
x = data[:, 1]
y = data[:, 2]
in_target = data[:, 3]

print("First times (s):", t[:5])

First times (s): [0.    0.022 0.027 0.03  0.037]


### Record times

In [40]:
# Number of complete records (a start AND an end)
n_records = min(len(starts), len(ends))

# Convert start and end times to seconds, with the same origin as t
t_starts = (np.array(starts) - data[0, 0]) / 1000
t_ends = (np.array(ends) - data[0, 0]) / 1000

print("record times (seconds since the first data point):")
for i in range(n_records):
    duration = t_ends[i] - t_starts[i]
    print(f"{t_starts[i]:7.3f} to {t_ends[i]:7.3f} ({duration:.3f} s)")

record times (seconds since the first data point):
  0.004 to  20.007 (20.003 s)
 40.011 to  60.022 (20.011 s)
 80.043 to 100.057 (20.014 s)
120.061 to 140.075 (20.014 s)
160.090 to 180.104 (20.014 s)


### Split the data into records

In [41]:
print("Number of records:", n_records)

# Lists to store the data of each record (used in the next tasks)
t_trials = []
x_trials = []
y_trials = []
in_target_trials = []

for i in range(n_records):
    # Keep only the samples between the start and the end of record i
    keep = (t >= t_starts[i]) & (t <= t_ends[i])

    t_trials.append(t[keep])
    x_trials.append(x[keep])
    y_trials.append(y[keep])
    in_target_trials.append(in_target[keep])

    # Number of samples, duration and sampling frequency
    t_rec = t[keep]
    n_samples = len(t_rec)
    duration = t_rec[-1] - t_rec[0]
    frequency = (n_samples - 1) / duration

    print(f"record {i}: {n_samples} samples from {t_rec[0]:7.3f} to {t_rec[-1]:7.3f} s, ({duration:.3f}s, {frequency:.3f} Hz)")

Number of records: 5
record 0: 3971 samples from   0.022 to  20.004 s, (19.982s, 198.679 Hz)
record 1: 3898 samples from  40.429 to  60.018 s, (19.589s, 198.938 Hz)
record 2: 3898 samples from  80.371 to 100.050 s, (19.679s, 198.028 Hz)
record 3: 3884 samples from 120.608 to 140.069 s, (19.461s, 199.527 Hz)
record 4: 3936 samples from 160.146 to 180.098 s, (19.952s, 197.223 Hz)
